# 🎬 Korean Dialogue Shorts — natural voice, one click

Runs the factory **in the cloud (open internet)**, so `edge-tts` makes a
**natural Korean neural voice** automatically — no robotic sound, no recording,
no local install.

**Run each cell top to bottom.** If something fails, the cell now prints the
full error so we can see exactly why.

> ⚠️ The GitHub repo must be **public** for Colab to clone it. If it's private,
> make it public (Settings → Danger Zone) or run on your own PC instead.


In [ ]:
#@title 1) Get the code + install (≈2 min)
import os, subprocess
if not os.path.isdir('/content/factory_repo'):
    r = subprocess.run('git clone --depth 1 -b claude/monetize-code-automation-e8tzee https://github.com/SEUNGMINSHIN7277/-.git /content/factory_repo',
                       shell=True, text=True, capture_output=True)
    print(r.stdout, r.stderr)
    if not os.path.isdir('/content/factory_repo'):
        raise SystemExit('❌ clone failed — is the repo PUBLIC? See the message above.')
%cd /content/factory_repo/content-factory
!pip -q install -r requirements.txt
!npm install
!npx playwright install --with-deps chromium
print('\n✅ install done')

In [ ]:
#@title 1b) Diagnostics (verify everything is ready)
import shutil, subprocess, sys, os
print('node  :', shutil.which('node'))
print('ffmpeg:', shutil.which('ffmpeg'))
print('factory import:', subprocess.run([sys.executable,'-c','import factory;print("OK")'],capture_output=True,text=True).stdout.strip())
os.makedirs('output', exist_ok=True)
open('/tmp/t.html','w').write("<div id=s style='width:200px;height:200px;background:#0a0'></div>")
r = subprocess.run(['node','tools/render_html.js','--html','/tmp/t.html','--out','/tmp/t.png','--width','200','--height','200','--selector','#s'],capture_output=True,text=True)
print('playwright smoke rc:', r.returncode, '| png:', os.path.exists('/tmp/t.png'))
if r.returncode: print('playwright stderr:\n', r.stderr[-1500:])
print('\n✅ diagnostics done' if os.path.exists('/tmp/t.png') else '\n❌ Playwright/Chromium not working — see stderr above')

In [ ]:
#@title 2) Options
pair = "dad_daughter" #@param ["", "dad_daughter", "mom_son", "couple", "grandma_grandchild", "grandpa_grandchild", "siblings"]
video_format = "call" #@param ["call", "scene"]
topic = "" #@param {type:"string"}
GEMINI_API_KEY = "" #@param {type:"string"}
#@markdown Optional FREE Azure Speech key → guarantees natural voice if the edge endpoint is rate-limited.
AZURE_SPEECH_KEY = "" #@param {type:"string"}
AZURE_SPEECH_REGION = "koreacentral" #@param {type:"string"}
print('options set')

In [ ]:
#@title 3) Make the video (shows full error if it fails)
import os, subprocess, sys
os.environ['VIDEO_FORMAT'] = video_format
os.environ['TTS_PROVIDER'] = 'edge'
os.environ['LLM_PROVIDER'] = 'gemini' if GEMINI_API_KEY else 'seed'
if GEMINI_API_KEY: os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
if AZURE_SPEECH_KEY:
    os.environ['TTS_PROVIDER'] = 'azure'
    os.environ['AZURE_SPEECH_KEY'] = AZURE_SPEECH_KEY
    os.environ['AZURE_SPEECH_REGION'] = AZURE_SPEECH_REGION
args = ['--format', video_format]
if pair: args += ['--pair', pair]
if topic: args += ['--topic', topic]
print('TTS:', os.environ['TTS_PROVIDER'], '| LLM:', os.environ['LLM_PROVIDER'], '| running...\n')
proc = subprocess.run([sys.executable,'-m','factory','make',*args], capture_output=True, text=True)
print(proc.stdout[-3500:])
if proc.returncode != 0:
    print('\n❌ FAILED (returncode', proc.returncode, ')\n--- STDERR ---\n', proc.stderr[-5000:])
else:
    print('\n✅ video created')

In [ ]:
#@title 4) Watch + download
import glob, os
from IPython.display import HTML, display
from base64 import b64encode
vids = sorted(glob.glob('output/**/*.mp4', recursive=True), key=os.path.getmtime)
assert vids, 'no video yet — fix the error printed in step 3 first'
v = vids[-1]; print('video:', v)
data = b64encode(open(v, 'rb').read()).decode()
display(HTML(f'<video width=320 controls autoplay src="data:video/mp4;base64,{data}"></video>'))
from google.colab import files; files.download(v)